In [19]:
import mosek
from mosek.fusion import *
import mosek.fusion.pythonic
import numpy as np
from math import sqrt
from sdp_valid_inequalities import weber
import csv

In [20]:
ineq_keys = [
    "I<P sdp bound",
    "diagonal bounds",
    "off-diagonal bounds",
    "2x2 minors",
    "triangle inequalities",
    "distance sym and nn",
    "pairwise distance lb",
    "eigencuts",
    "sparse eigencuts",
    "sparse dnn eigencuts",
    "boros hammer",
    "clique inequalities",
    "odd-cycle inequalities",
    "socrlt perspective cuts",
    "rlt assignment consistency",
    "distance projection linking cuts",
    "geometric conflict cuts",
    "distance square diameter bounds",
    "cluster symmetry break"
]

In [21]:
demands = [[0,0],[1,0],[0,1]]
p = 2

experiments = []

In [28]:
best = {
    "cluster symmetry break" : True,
    "distance square diameter bounds" : True
}

In [29]:
for n in range(-1,len(ineq_keys)):
    ineq = dict(zip(ineq_keys,[n==i for i in range(len(ineq_keys))])) | best

    alreadyDone = False
    for exp in experiments:
        same = True
        for key in ineq_keys:
            if ineq[key] != exp[key]:
                same = False
                break
        if same:
            alreadyDone = True
            break
    if alreadyDone and len(experiments) > 0:
        continue
    
    sol,info = weber(demands,p,ineq)

    score = 0
    for j in range(p):
        score += np.linalg.eigh(sol["P"][j]).eigenvalues[-1]
    score /= p

    BDPnorm = 0
    for j in range(p):
        BDPnorm += np.linalg.norm(sol["B"][j] - (sol["D"][j]@sol["P"][j]))
    BDPnorm /= p

    Pnorm = 0
    for j in range(p):
        Pnorm += np.linalg.norm(sol["P"][j])
    Pnorm /= p

    n = len(demands)
    dim = len(demands[0])
    mean = sum([sol["y"][j,:] for j in range(p)])/p
    var = sum([np.linalg.norm(sol["y"][j,:] - mean)**2 for j in range(p)])/p

    row = ineq | info | {
        "eigenscore" : score,
        "BDPnorm" : BDPnorm,
        "Pnorm" : Pnorm,
        "centerVar" : var
    }
    experiments.append(row)

In [30]:
with open("table.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(experiments[0].keys())
    writer.writerows([exp.values() for exp in experiments])